In [1]:
import random, time, uuid, httpx, asyncio, json
from datetime import datetime, timezone
from sqlalchemy import create_engine, select, update
from sqlalchemy.orm import sessionmaker

# Import your database tables from your database module
from utils.database import EventSource, RawEvent
from utils.schemas import RawEventRead

# 🌟 Configure the clean synchronous DB Engine (Using standard sqlite driver)
DATABASE_URL = 'sqlite:///./data/database.db'
engine = create_engine(DATABASE_URL, connect_args={'check_same_thread': False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

In [2]:
ID = '592f314e-38a1-4034-ae1f-a232c77264bf'
with SessionLocal() as session:
    result = session.execute(select(RawEvent))
    events = result.scalars().all()
    events = [RawEventRead.model_validate(e).model_dump() for e in events]

In [5]:
adds = [e['address'] for e in events]

In [6]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="socal_geocoder")

socal_viewbox = [
    (35.8, -120.5),  # northwest
    (32.5, -114.1),  # southeast
]

def address2coordinates(address: str) -> tuple[float, float] | None:
    try:
        location = geolocator.geocode(
            address,
            viewbox=socal_viewbox,
            bounded=True,
            country_codes="us",
        )

        if location is None:
            return 0,0

        return location.latitude, location.longitude
    except Exception as e:
        print(f'[geospatial:exception] {e}')
        return 0,0

In [12]:
adds_enhanced = [f'{a}, Los Angeles' for a in adds]

In [13]:
for i,add in enumerate(adds_enhanced):
    print(i,add,address2coordinates(add))

0 South of Conant +53, Los Angeles (0, 0)
1 Seoul International Park +30, Los Angeles (0, 0)
2 Veterans Park, Los Angeles (34.0559229, -117.7865004)
3 MG Studio $20+, Los Angeles (0, 0)
4 1625 17th St unit 6 Near Capacity, Los Angeles (0, 0)
5 Venice Sign +40, Los Angeles (0, 0)
6 Santa Monica, CA, Los Angeles (34.0194704, -118.491227)
7 2415 Michigan Ave +276, Los Angeles (34.0449699, -118.2079236)


In [16]:
address2coordinates('MG Studio')

(0, 0)